# APEX Optimizer Tutorial: Math Problem Solving

This tutorial demonstrates how to use **APEX** (Analysis-based Prompt Engineering eXpert) to improve GPT-5 Mini's performance on AIME math problems through systematic prompt optimization.

APEX analyzes failures, recognizes success patterns, generates hypotheses, and validates improvements empirically.

## Configuration

All modifiable parameters in one place for easy adjustment:

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Pydantic serializer warnings:",
    category=UserWarning,
    module="pydantic.main",
)

In [2]:
from dspy.teleprompt.apex.litellm_session_pool import set_pool_size_for_litellm_session

# API Configuration
api_key = 'sk-12345'# Will prompt if not set
base_url = 'https://nexus-master.lmndstaging.com'

# Student Model Configuration (model being optimized)
student_model = "litellm_proxy/openai/gpt-5-mini"
student_base_url = base_url  # Optional custom API endpoint
student_temperature = 0.0  # Deterministic for math
student_reasoning_effort = 'minimal'

# Analysis Model Configuration (for failure analysis and hypotheses)
# analysis_model = "litellm_proxy/openai/gpt-5"
analysis_model = "litellm_proxy/vertex_ai/gemini-2.5-pro"
analysis_base_url = base_url  # Optional custom API endpoint
analysis_temperature = 1.0  # Creative for hypothesis generation
analysis_reasoning_effort = None#'minimal'

# APEX Optimization Settings
max_iterations = 50
num_hypotheses = 3
train_sample_size = 10
success_threshold = 1.0
convergence_patience = 10
num_threads = 50
seed = 42
verbosity = "detailed"
candidate_selection = "pareto"

# MLflow Tracking (Optional)
use_mlflow = True
mlflow_tracking_uri = "http://localhost:5005"
mlflow_experiment_name = "APEX-AIME-Math"

set_pool_size_for_litellm_session(pool_size=num_threads)

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

Import dependencies and configure language models:

In [3]:
import os
import dspy
from dspy.adapters import JSONAdapter

if api_key is None:
    api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")

# Configure student model
student_kwargs = {
    "model": student_model,
    "api_key": api_key,
    "temperature": student_temperature,
}
if student_base_url:
    student_kwargs["base_url"] = student_base_url
if student_reasoning_effort:
    student_kwargs["reasoning_effort"] = student_reasoning_effort

student_lm = dspy.LM(**student_kwargs)

# Configure analysis model
analysis_kwargs = {
    "model": analysis_model,
    "api_key": api_key,
    "temperature": analysis_temperature,
}
if analysis_base_url:
    analysis_kwargs["base_url"] = analysis_base_url
if analysis_reasoning_effort:
    analysis_kwargs["reasoning_effort"] = analysis_reasoning_effort

analysis_lm = dspy.LM(**analysis_kwargs)

analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=num_threads)

## Dataset

Load AIME problems (American Invitational Mathematics Examination):

In [4]:
from datasets import load_dataset
import random

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

train_set, val_set, test_set = init_dataset()

print(f"Training: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

Training: 45 | Validation: 45 | Test: 150


Example problem:

In [5]:
print("Problem:", train_set[0]['problem'])
print("\nAnswer:", train_set[0]['answer'])

Problem: In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.

Answer: 242


## Program

Define a Chain of Thought program:

In [6]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

## Metrics

Define evaluation metrics:

In [7]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [8]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer. "
            f"You responded with '{prediction.answer}', which couldn't be parsed. "
            f"The correct answer is '{correct_answer}'."
        )
        
        if written_solution:
            feedback_text += f" Here's the full solution:\n{written_solution}"
        
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    
    if score == 1:
        feedback_text = f"Correct! The answer is '{correct_answer}'."
    else:
        feedback_text = f"Incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full solution:\n{written_solution}"

    return dspy.Prediction(score=score, feedback=feedback_text)

## Baseline Evaluation

Evaluate the unoptimized program:

In [9]:
eval_kwargs = dict(
    num_threads=num_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

print("Evaluating baseline...")
baseline_result = evaluate(program)

print(f"\nBaseline Performance: {baseline_result.score / 100.:.1%}")

Evaluating baseline...
Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:00<00:00, 267.50it/s]

2025/10/18 20:12:59 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]



Baseline Performance: 53.3%


## APEX Optimization

Optimize the program with APEX:

In [10]:
from tqdm.contrib.logging import logging_redirect_tqdm
from dspy.teleprompt.apex import APEX

optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,
    hypothesis_lm=analysis_lm,
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=max_iterations,
    num_hypotheses=num_hypotheses,
    num_eval_runs=1,
    train_sample=train_sample_size,
    success_threshold=success_threshold,
    convergence_patience=convergence_patience,
    num_threads=num_threads,
    verbosity=verbosity,
    seed=seed,
    candidate_selection=candidate_selection,
    use_mlflow=use_mlflow,
    mlflow_tracking_uri=mlflow_tracking_uri,
    mlflow_experiment_name=mlflow_experiment_name,
)

print("Starting optimization...")

with logging_redirect_tqdm():
    optimized_program = optimizer.compile(
        student=program,
        trainset=train_set,
        valset=val_set,
    )

print("\nOptimization complete!")

2025/10/18 20:12:59 INFO dspy.teleprompt.apex.apex: APEX: MLflow tracking enabled


Starting optimization...


2025/10/18 20:12:59 INFO dspy.teleprompt.apex.apex: APEX: running with num_threads=50
2025/10/18 20:12:59 INFO dspy.teleprompt.apex.apex: APEX: Configuration - max_iterations=50, num_hypotheses=3, success_threshold=1.00, convergence_patience=10
2025/10/18 20:12:59 INFO dspy.teleprompt.apex.apex: APEX: Using seed=42 for reproducibility
2025/10/18 20:12:59 INFO dspy.teleprompt.apex.apex: APEX: Evaluating initial baseline on validation set


Processed 45 / 45 examples: 100%|██████████| 45/45 [00:01<00:00, 30.74it/s]

2025/10/18 20:13:00 INFO dspy.teleprompt.apex.apex: APEX: Initial baseline score=0.5111


...

2025/10/18 22:01:09 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.
2025/10/18 22:01:09 INFO dspy.teleprompt.apex.apex: APEX: Optimization interrupted by user (Ctrl+C)


Processed 7 / 10 examples:  70%|███████   | 7/10 [01:19<00:33, 11.31s/it]

2025/10/18 22:01:09 INFO dspy.teleprompt.apex.apex: APEX: ✓ Optimization complete | 17 iterations | Reason: interrupted
2025/10/18 22:01:09 INFO dspy.teleprompt.apex.apex: APEX: Final score: 0.7111 (+0.2000 from baseline 0.5111)
2025/10/18 22:01:09 INFO dspy.teleprompt.apex.apex: APEX: Summary - evaluated 68 candidates from 51 hypotheses
2025/10/18 22:01:09 INFO dspy.teleprompt.apex.apex: APEX: Best score trajectory across iterations: [0.5333333333333333, 0.5111111111111111, 0.6444444444444445, 0.6444444444444445, 0.6222222222222222, 0.6666666666666666, 0.6222222222222222, 0.7111111111111111, 0.6, 0.6222222222222222, 0.6666666666666666, 0.6666666666666666, 0.6, 0.5777777777777777, 0.6444444444444445, 0.6444444444444445, 0.6666666666666666]



🏃 View run respected-snake-705 at: http://localhost:5005/#/experiments/1/runs/7e81a5c1c076455a8a80431ae36edf4c
🧪 View experiment at: http://localhost:5005/#/experiments/1

╔═══════════════════════════════════════════════════════════════╗
║                    APEX Optimization Summary                     ║
╠═══════════════════════════════════════════════════════════════╣
║ Total Iterations: 17                                              ║
║ Hypotheses Tested: 51                                             ║
║ Candidates Evaluated: 68                                          ║
║ Selection Strategy: pareto                                    ║
║ Pareto Merge Probability: 0.00                             ║
╠═══════════════════════════════════════════════════════════════╣
║ Initial Score: 0.5111                                             ║
║ Final Score: 0.7111                                               ║
║ Improvement: +0.2000                                              ║
╠══════════

[Trace(trace_id=tr-401cb123ff96499ee481707dd08f49c5), Trace(trace_id=tr-0b01cab6e1b32eec42f748f38200c825), Trace(trace_id=tr-b08e0a62559017acc77dc1030c78738c), Trace(trace_id=tr-581b99bceb412478adbbb1d88ec367f1), Trace(trace_id=tr-e84c6d66c07bf61a47968d76a9ad332c), Trace(trace_id=tr-ff7b21cee78b723fd6691310c204cdcc), Trace(trace_id=tr-aadf789d3fd463d11b2a89380299513c), Trace(trace_id=tr-0ebee3a11f011b2de269863485b3a269), Trace(trace_id=tr-1069cea995938dbbd1af7eb9add2b19d), Trace(trace_id=tr-798df72e28ccd314d0375af90eeaf4b8)]

Inspect the optimized prompt:

In [11]:
print("Optimized Prompt:")
print("=" * 50)
print(optimized_program.predict.signature.instructions)
print("=" * 50)

Optimized Prompt:
Instructions: Instructions: Solve the problem by following these steps: Plan, Execute, and Verify.

**1. Plan:**
*   First, carefully analyze the problem to identify the core question and all constraints. **Always check for simplifications first (e.g., comparing exponents) before selecting a complex strategy.**
*   Identify the type of problem and outline a solution strategy. **If a problem mentions a 'unique' point or object, treat this as a strong clue that a special geometric property or symmetry exists. Identify this property to simplify the problem before attempting complex calculations.** **If an initial path appears to lead to extreme complexity, pause and double-check your interpretation for a simpler approach before proceeding. However, if a path leads to a contradiction or produces results that conflict with problem constraints, you MUST discard the entire path. You are FORBIDDEN from inventing an unsubstantiated formula as a shortcut. Instead, you must re-e

[Trace(trace_id=tr-de2cda6be8b305c8426878184b65692d), Trace(trace_id=tr-dde75e61bae8974848083ddcece06204), Trace(trace_id=tr-60019ff91932fbf3556fcbaf6d6d17eb)]

## Final Evaluation

Evaluate the optimized program:

In [12]:
print("Evaluating optimized program...")
optimized_result = evaluate(optimized_program)

print(f"\n{'='*50}")
print(f"Baseline:  {baseline_result.score/100.:.1%}")
print(f"Optimized: {optimized_result.score/100.:.1%}")
print(f"Improvement: {(optimized_result.score - baseline_result.score)/100.:.1%}")
print(f"{'='*50}")

Evaluating optimized program...
Average Metric: 91.00 / 130 (70.0%):  87%|████████▋ | 130/150 [01:28<00:34,  1.71s/it]

2025/10/18 22:03:28 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Let $ABCDE$ be a convex pentagon with $AB=14$, $BC=7$, $CD=24$, $DE=13$, $EA=26$, and $\\angle B=\\angle E=60^{\\circ}$. For each point $X$ in the plane, define $f(X)=AX+BX+CX+DX+EX$. The least possible value of $f(X)$ can be expressed as $m+n\\sqrt{p}$, where $m$ and $n$ are positive integers and $p$ is not divisible by the square of any prime. Find $m+n+p$.', 'answer': 60}) (input_keys={'problem'}): Adapter JSONAdapter failed to parse the LM response. 

LM Response: {
  "reasoning": "Plan:\nWe are given a convex pentagon ABCDE with side lengths AB=14, BC=7, CD=24, DE=13, EA=26 and angles at B and E equal to 60 degrees. For any point X define f(X)=AX+BX+CX+DX+EX. We must find the minimal possible f(X) over the plane. This is a classical Fermat/Steiner-type problem: the sum of distances from a variable point to fixed points is minimized. For a set of points, if we reflect some of the points across lines a

Average Metric: 100.00 / 149 (67.1%): : 151it [03:31,  1.40s/it]                       

2025/10/18 22:05:30 INFO dspy.evaluate.evaluate: Average Metric: 100.0 / 150 (66.7%)


,problem,example_answer,reasoning,pred_answer,metric,answer
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70.0,Plan: - Interpret numbers in base b. 17_b means digits 1 and 7 so ...,70,✔️ [1.000],NaN
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588.0,Plan: - We have triangle ABC with points on AB: A-D-E-B in order w...,588,✔️ [1.000],NaN
2,The 9 members of a baseball team went to an ice-cream parlor after...,16.0,Plan: We have 9 labeled players; each chooses one of 3 flavors: ch...,16,✔️ [1.000],NaN
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117.0,"Plan: We need integer solutions (x,y) with -100 ≤ x,y ≤ 100 satisf...",117,✔️ [1.000],NaN
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279.0,Plan: - We must count 8-digit permutations of digits 1..8 divisibl...,279,✔️ [1.000],NaN



Baseline:  53.3%
Optimized: 66.7%
Improvement: 13.3%


## Optimization Insights

Examine the optimization process:

In [13]:
if hasattr(optimized_program, 'apex_result'):
    result = optimized_program.apex_result
    
    print("Summary:")
    print(f"  Iterations: {len(result.iterations)}")
    print(f"  Candidates evaluated: {len(result.all_candidates)}")
    print(f"  Stop reason: {result.stopped_after}")
    print(f"  Best score: {result.best_candidate.overall_score:.4f}")
    
    print("\nIteration Progress:")
    for it in result.iterations:
        print(f"  Iteration {it.iteration}: {it.num_failures} failures, {len(it.hypotheses)} hypotheses, {len(it.candidates)} candidates")
    
    if result.best_candidate.hypothesis:
        h = result.best_candidate.hypothesis
        print(f"\nBest Hypothesis:")
        print(f"  Strategy: {h.strategy if hasattr(h, 'strategy') else 'N/A'}")
        print(f"  Impact Score: {h.impact_score if hasattr(h, 'impact_score') else 'N/A'}")

Summary:
  Iterations: 17
  Candidates evaluated: 69
  Stop reason: interrupted
  Best score: 0.7111

Iteration Progress:
  Iteration 1: 3 failures, 3 hypotheses, 4 candidates
  Iteration 2: 3 failures, 3 hypotheses, 4 candidates
  Iteration 3: 3 failures, 3 hypotheses, 4 candidates
  Iteration 4: 5 failures, 3 hypotheses, 4 candidates
  Iteration 5: 4 failures, 3 hypotheses, 4 candidates
  Iteration 6: 3 failures, 3 hypotheses, 4 candidates
  Iteration 7: 2 failures, 3 hypotheses, 4 candidates
  Iteration 8: 2 failures, 3 hypotheses, 4 candidates
  Iteration 9: 3 failures, 3 hypotheses, 4 candidates
  Iteration 10: 3 failures, 3 hypotheses, 4 candidates
  Iteration 11: 4 failures, 3 hypotheses, 4 candidates
  Iteration 12: 4 failures, 3 hypotheses, 4 candidates
  Iteration 13: 2 failures, 3 hypotheses, 4 candidates
  Iteration 14: 5 failures, 3 hypotheses, 4 candidates
  Iteration 15: 4 failures, 3 hypotheses, 4 candidates
  Iteration 16: 4 failures, 3 hypotheses, 4 candidates
  Itera

## Conclusion

APEX systematically optimizes prompts through:
1. Analyzing failures to understand root causes
2. Recognizing successful patterns
3. Generating data-driven hypotheses
4. Validating improvements empirically

Try adjusting the configuration parameters to explore different optimization strategies.